In [ ]:
import numpy as np
import torch
import torch.nn as nn

import h5py
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import csv

import tempfile
import os
import atexit
import glob
import shutil
from pathlib import Path

from src.model_utils import setup_model
from src.config_utils import setup_config_variables

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

print("This can be run after preprocessing.")

In [ ]:
config = setup_config_variables()

In [ ]:
# SETUP MODEL
model = setup_model(config["MODEL_TYPE"], config["MODEL_DIMS"], device)

In [ ]:
model_files = []
best_model_dir = f"{config['MODEL_SAVE_DIR']}/best/*.pth"
last_model_dir = f"{config['MODEL_SAVE_DIR']}/last/*.pth"

# pull best and last from a copy from tmpdir, in case they change while plotting (i.e. training is going on in parallel)
tmpdir = tempfile.mkdtemp()
atexit.register(lambda: shutil.rmtree(tmpdir, ignore_errors=True))
for path in glob.glob(best_model_dir):
    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))
for path in glob.glob(last_model_dir):
    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))
    
# first best, then last
model_files.extend(sorted(glob.glob(tmpdir + "/*.pth"), key=os.path.getctime))

print(f"{model_files}")
train_files = sorted(glob.glob(config['TRAINING_DIR'] + "/*.hdf5"))
test_files = sorted(glob.glob(config['TEST_DIR'] + "/*.hdf5"))
whole_files = sorted(glob.glob(config['OUT_DIR'] + "/*.hdf5"))

In [ ]:
# Delete old images
try:
        os.mkdir(config["IMAGES_DIR"])
except FileExistsError:
    for img in glob.glob(config["IMAGES_DIR"] + "/*.svg"):
        os.remove(img)

In [ ]:
criterion = nn.MSELoss()
steering_slope = 1 / 180  # (output_max - output_min)/(input_max - input_min) = (1-(-1))/(180-(-180))
inv_steering_slope = 180

def generate_predictions(model, file_path, batch_size=config["BATCH_SIZE"], window_size=config["WINDOW_SIZE"]):
    inv_apsdvs_max = 1 / 255
    def normalise(data):
            # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
            
            return (data * inv_apsdvs_max).astype(config["DTYPE"])

    # used for getting the data out at the end.
    def denormalise_steering(data):
        return data * inv_steering_slope
    
    model.eval()

    with h5py.File(file_path, "r") as f:
        dvs_frame = np.array(f[config["DVS_FRAME_NAME"]])
        data_len = dvs_frame.shape[0]
        aps_frame = np.array(f[config["APS_FRAME_NAME"]])
        steering_angles = np.array(f["steering_wheel_angle"])
        if config["USE_APS"] and config["USE_DVS"]:
            # we have batch x 224 x 224 x 2 after stack
            combined_frames = np.stack((dvs_frame, aps_frame), axis=3)
        else:
            combined_frames = aps_frame if config["USE_APS"] else dvs_frame
            combined_frames = np.expand_dims(combined_frames, axis=3)
        
        preds, gts = [], []
        batched_window_size = batch_size * window_size
        for start in range(0, data_len, batched_window_size):
            end = min(start + batched_window_size, data_len)
            batch_data = combined_frames[start:end]

            # if the batch has fewer frames than a single window, skip processing
            if batch_data.shape[0] < window_size:
                continue
            # Only complete windows are processed; discard leftover frames
            num_windows = batch_data.shape[0] // window_size
            batch_data = batch_data[: num_windows * window_size]

            batch_data = normalise(batch_data)
            # we have batch_size * window_size, h, w, c
            # we want batch_size, window_size, c, h, w
            batch_data = batch_data.reshape(num_windows, window_size, *batch_data.shape[1:])
            # shuffle dimensions
            batch_data = batch_data.transpose(0, 1, 4, 2, 3)
            
            batch_data = torch.from_numpy(batch_data).to(device)

            with torch.no_grad():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    batch_out = model(batch_data).cpu().numpy()
                    # if windowed and batched, output will be 3 dimensional like b, w, 1
                    # if not windowed, it will be 2 dimensional like b, 1
                    # we want to reshape to 1 dimension.
                    batch_out = batch_out.squeeze(-1).flatten()

                batch_out = denormalise_steering(batch_out)
                preds.append(batch_out)
                if model in ["Resnet"]:
                    gts.append(steering_angles[start:start + num_windows * window_size])
                
                # only add last frame
                else:
                    gts.append(steering_angles[start: start + num_windows * window_size: window_size])
       
        preds = np.concatenate(preds, axis=0)
        gts = np.concatenate(gts, axis=0)

        # Convert to torch tensors
        preds_tensor = torch.tensor(preds).to(device)
        gts_tensor = torch.tensor(gts).to(device)

        # Calculate loss
        loss = criterion(preds_tensor, gts_tensor)

        return preds, loss

In [ ]:
def plot_predictions(model, files, model_files, dataset_name):
    fig = plt.figure(figsize=(15, 7.5 * len(files)))
    for i, f in enumerate(files):
        announce_str = f"Working on {f}, File [{i + 1}/{len(files)}]"
        print(f"{announce_str}\n{'-' * len(announce_str)}")
        ax = plt.subplot(len(files), 1, i + 1)
        ax.set_title(f, fontsize=14)
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel("Predicted Values", fontsize=12)
        ax.grid(True)
        ax.set_ylim(-90, 90)
        ax.xaxis.set_major_locator(MultipleLocator(25))
        ax.tick_params(axis='x', rotation=45)
        with h5py.File(f, "r") as f5:
            gt = np.array(f5["steering_wheel_angle"])
        if len(gt) == 0:
            print("Empty dataset! Skipping.")
            continue
        time_axis = [x * (config["BINSIZE_MS"] / 1000) for x in range(len(gt))]
        ax.plot(time_axis, gt, label="truth", linewidth=2, linestyle='--')
        
        for model_file in model_files:
            epoch = Path(model_file).stem.split('_')[0]
            epoch = epoch.split('/')[-1][1:]
            loc = f"cuda:{0}"
            model_dict = torch.load(model_file, weights_only=False, map_location=loc)
            
            new_state_dict = {}
            for key, value in model_dict["model_state_dict"].items():
                new_state_dict[key.replace("module.", "")] = value
                
            model.load_state_dict(new_state_dict)
            model.to(device)

            preds, loss = generate_predictions(model, f)

            # if preds is shorter than gt (because e.g. one pred is made per 8 ground truths due to sequence size being 8) elongate preds.

            seq_len = len(gt) // len(preds)
            preds_stretched = np.repeat(preds, seq_len)
            # it's ok if it's still a bit short (< len(preds))


            # new time axis, it might be shorter since windows are considered.
            time_axis = [x * (config["BINSIZE_MS"] / 1000) for x in range(len(preds_stretched))]
            print(f"RMSE for model {epoch} is {loss ** 0.5:.2f}")
            ax.plot(time_axis, preds_stretched, label=f"E:{epoch}, L:{loss ** 0.5:.2f}")
            stretched_loss = criterion(
                torch.tensor(preds_stretched, dtype=torch.float32).to(device),
                torch.tensor(gt[:len(preds_stretched)], dtype=torch.float32).to(device)
            )
            print(f"RMSE for model {epoch} with stretched preds is {stretched_loss ** 0.5:.2f}")

        l = ax.legend(loc='best')
        if len(model_files) == 0:
            print("No models found! just plotting ground truth.")
        
        else:
            # add the "best" and "last" to legend
            # edge case, best and last are the same
            if len(l.get_texts()) > 2:
                current_text = l.get_texts()[-2].get_text()
                l.get_texts()[-2].set_text(f"Best: {current_text}")
                current_text = l.get_texts()[-1].get_text()
                l.get_texts()[-1].set_text(f"Last: {current_text}")
            else:
                current_text = l.get_texts()[-1].get_text()
                l.get_texts()[-1].set_text(f"Best/Last: {current_text}")

    plt.tight_layout()
    svg_path = os.path.join(config["IMAGES_DIR"], f"{dataset_name}.svg")
    fig.savefig(svg_path, format="svg")
    print(f"Saved plot to {svg_path}")
    plt.show()
    plt.close(fig)

In [ ]:
# From Claude 3.7 Sonnet Thought
import time
from tqdm import tqdm

# Add this cell to measure inference time
import time
from tqdm import tqdm

def measure_inference_time(model, test_file, batch_size=config["BATCH_SIZE"], 
                          window_size=config["WINDOW_SIZE"], num_runs=10):
    """
    Measures inference time statistics for the model.
    
    Args:
        model: The PyTorch model to benchmark
        test_file: Path to a test HDF5 file
        batch_size: Batch size to use
        window_size: Window size for sequential models
        num_runs: Number of times to repeat the measurement for better statistics
    
    Returns:
        Dictionary with timing statistics and raw inference times
    """
    model.eval()
    
    # Load a representative batch from the test file
    inv_apsdvs_max = 1 / 255
    
    with h5py.File(test_file, "r") as f:
        dvs_frame = np.array(f[config["DVS_FRAME_NAME"]])
        aps_frame = np.array(f[config["APS_FRAME_NAME"]])
        
        if config["USE_APS"] and config["USE_DVS"]:
            combined_frames = np.stack((dvs_frame, aps_frame), axis=3)
        else:
            combined_frames = aps_frame if config["USE_APS"] else dvs_frame
            combined_frames = np.expand_dims(combined_frames, axis=3)
            
        # Get enough frames for one batch
        batch_data = combined_frames[:batch_size * window_size]
        
        # Normalize and reshape as in generate_predictions function
        batch_data = (batch_data * inv_apsdvs_max).astype(config["DTYPE"])
        num_windows = batch_data.shape[0] // window_size
        batch_data = batch_data[:num_windows * window_size]
        batch_data = batch_data.reshape(num_windows, window_size, *batch_data.shape[1:])
        batch_data = batch_data.transpose(0, 1, 4, 2, 3)
        
        batch_data_tensor = torch.from_numpy(batch_data).to(device)
        
        # Prepare input based on model type
        if config["MODEL_TYPE"] == "Resnet":
            # Squeeze the sequence dimension
            b, t, c, w, h = batch_data_tensor.shape
            batch_data_tensor = batch_data_tensor.view(b * t, c, w, h)
        elif config["MODEL_TYPE"] != "ResnetRNN":
            batch_data_tensor = batch_data_tensor.permute(0, 2, 1, 3, 4)
    
    # Warm up GPU
    print("Warming up...")
    with torch.no_grad():
        for _ in range(5):
            _ = model(batch_data_tensor)
    
    # Measure inference time
    print(f"Measuring inference time over {num_runs} runs...")
    inference_times = []
    
    with torch.no_grad():
        for _ in tqdm(range(num_runs)):
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            start_time = time.time()
            
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                _ = model(batch_data_tensor)
                
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            end_time = time.time()
            inference_times.append(end_time - start_time)
    
    # Calculate statistics
    total_frames = batch_size * window_size
    
    inference_times_array = np.array(inference_times)
    avg_time = np.mean(inference_times_array)
    std_time = np.std(inference_times_array)
    
    stats = {
        "avg_time_per_batch_ms": avg_time * 1000,
        "std_time_ms": std_time * 1000,
        "avg_time_per_frame_ms": (avg_time * 1000) / total_frames,
        "frames_per_second": total_frames / avg_time,
        "batch_size": batch_size,
        "window_size": window_size,
        "total_frames": total_frames
    }
    
    return stats, inference_times_array

# Make sure you have a model loaded (if running this separately)
# model = setup_model(config["MODEL_TYPE"], config["MODEL_DIMS"], config["PRETRAINED"], device)

# Load the best model
if model_files:
    loc = "cuda:0" if torch.cuda.is_available() else "cpu"
    model_dict = torch.load(model_files[0], weights_only=False, map_location=loc)
    new_state_dict = {key.replace("module.", ""): value for key, value in model_dict["model_state_dict"].items()}
    model.load_state_dict(new_state_dict)
    model.to(device)

# Choose a representative test file
test_file = test_files[0] if test_files else train_files[0]

# Measure and display inference time statistics
stats, inference_times = measure_inference_time(model, test_file,num_runs=100)
print("\nInference Time Statistics:")
print(f"Model: {config['MODEL_TYPE']}")
print(f"Device: {device}")
print(f"Batch size: {stats['batch_size']}, Window size: {stats['window_size']}")
print(f"Average time per batch: {stats['avg_time_per_batch_ms']:.2f} ms (±{stats['std_time_ms']:.2f} ms)")
print(f"Average time per frame: {stats['avg_time_per_frame_ms']:.2f} ms")
print(f"Throughput: {stats['frames_per_second']:.2f} frames per second")

# Plot timing distribution
plt.figure(figsize=(10, 6))
plt.hist(inference_times * 1000, bins=20, alpha=0.7, color='blue')
plt.axvline(stats['avg_time_per_batch_ms'], color='red', linestyle='dashed', linewidth=2)
plt.xlabel('Inference Time (ms)')
plt.ylabel('Frequency')
plt.title('Distribution of Inference Times')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(config["IMAGES_DIR"], "inference_time_distribution.svg"), format="svg")
plt.show()

In [ ]:
def visualize_losses(log_dir):
    log_file = f"{log_dir}/loss_log.csv"
    epochs = []
    train_losses = []
    test_losses = []

    with open(log_file, "r") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            epoch, train_loss, test_loss = int(row[0]), float(row[1]), float(row[2])
            epochs.append(epoch)
            train_losses.append((train_loss ** 0.5) * 180)
            test_losses.append((test_loss ** 0.5) * 180)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label="Train Loss", marker='o', color="blue")
    plt.plot(epochs, test_losses, label="Test Loss", marker='x', color="red")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Testing Loss")
    plt.legend()
    plt.grid(True)
    plt.ylim(bottom=0, top=max(70, max(max(train_losses), max(test_losses))))
    plt.show()


visualize_losses(config["MODEL_SAVE_DIR"])

In [ ]:
# plot every 20th one to save time
plot_predictions(model, test_files[::20], model_files, "Test")

In [ ]:
# plot every 20th one to save time
plot_predictions(model, train_files[::20], model_files, "Train")

In [ ]:
model = None
# don't need to model performance on these, just seeing the data is ok.
plot_predictions(model, whole_files[:int(len(whole_files) / 3):1], model_files, "Unedited1of3")
plot_predictions(model, whole_files[int(len(whole_files) / 3):2 * int(len(whole_files) / 3):1], model_files, "Unedited2of3")
plot_predictions(model, whole_files[2 * int(len(whole_files) / 3)::1], model_files, "Unedited3of3")